# Dutch government publication RAG smoke test

This notebook validates the first Dutch government publication development corpus for Stage 230. It uses `stb-2022-14.pdf`, a Staatsblad publication containing the Wet open overheid text placement, and reuses the same RAG runtime that already passed the AG News compatibility test: Llama Stack, pgvector, Qwen3 reranking, and Nemotron through Models-as-a-Service.


## 1. Confirm runtime configuration and sample metadata


In [ ]:
import json
import os
from pathlib import Path
from llama_stack_client import LlamaStackClient

stage = Path.cwd() / '.stage230'
chunks = stage / 'data/dutch-government/processed/stb-2022-14-chunks.jsonl'
questions = stage / 'data/dutch-government/processed/stb-2022-14-questions.json'
source_pdf = stage / 'data/dutch-government/source/stb-2022-14.pdf'
assert chunks.exists(), 'missing Dutch publication chunks'
assert questions.exists(), 'missing Dutch publication questions'
assert source_pdf.exists(), 'missing source PDF'

for name in [
    'LLAMA_STACK_BASE_URL',
    'RHOAI_STAGE230_RERANKER_BASE_URL',
    'RHOAI_STAGE230_GENERATION_BASE_URL',
    'RHOAI_STAGE230_EMBEDDING_MODEL',
    'RHOAI_STAGE230_GENERATION_MODEL',
    'RHOAI_STAGE230_RERANKER_MODEL',
]:
    print(f'{name}={os.environ.get(name)}')
print('Llama Stack client import:', LlamaStackClient.__name__)
print('Source PDF:', source_pdf)
question_payload = json.loads(questions.read_text(encoding='utf-8'))
print('Vector store:', question_payload['vector_store'])
for item in question_payload['questions']:
    print(f"- {item['id']}: {item['question']}")


## 2. Run the Dutch publication smoke test

This creates or resets the `stage230-dutch-woo-demo` vector store, uploads article-level chunks from `stb-2022-14.pdf`, extracts a metadata topic filter from the Dutch question, runs filtered hybrid retrieval, reranks candidates, and asks Nemotron to answer in the user question language.


In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    '.stage230/scripts/dutch_publication_rag_smoke.py',
    '--reset',
    '--vector-store', 'stage230-dutch-woo-demo',
    '--search-mode', 'hybrid',
    '--query', 'Binnen welke termijn moet een bestuursorgaan beslissen op een verzoek om informatie?',
    '--expected-topic', 'openbaarmaking_op_verzoek',
    '--expected-term', 'vier weken',
], check=True)


The same helper can run the additional questions listed above by changing `--query`, `--expected-topic`, and `--expected-term`. For the larger corpus phase, keep this smoke document as a regression test and add Docling/KFP automation before indexing the full corpus.
